In [161]:
import networkx as nx

In [162]:
def plot_grafo(graphs, nombres=None, num_col_plot=5):
    if nombres == None:
        nombres = ["G_base"]
        for i in range(len(graphs) - 1):
            nombres.append("G"+ str(i + 1))
            
    if isinstance(graphs, nx.DiGraph) or isinstance(graphs, nx.MultiDiGraph):
        graphs = [graphs]
    if not isinstance(graphs, list):
        print("Error, only accepted arrays of DiGraphs or DiGraphs")
        return


    plt.style.use('seaborn-v0_8-darkgrid')
    num_row = int(len(graphs)/num_col_plot) + 1
    fig, ax_arr = plt.subplots(num_row, num_col_plot)
    if len(graphs) == 1:
        ax_arr = [ax_arr]
    ax_arr = ax_arr.flat
    fig.set_size_inches(16, num_row*4)

    for i, G in enumerate(graphs):
        pos = nx.spring_layout(G, seed=42)
        # We locate to the leftmost part the first node
        pos[min(pos.keys())] = [-1.5, 0]

        # The layout is adjusted
        for node in pos:
            if node != 0:
                pos[node][0] += 0.5  # Move all nodes to the right



        for node, data in G.nodes(data=True):
            if data["type"] == "process":
                nx.draw_networkx_nodes(G, pos, nodelist=[node], node_shape='s', node_color='lightblue', edgecolors='black', linewidths=1.5, ax=ax_arr[i])
            elif data["type"] == "file":
                nx.draw_networkx_nodes(G, pos, nodelist=[node], node_shape='o', node_color='lightgreen', edgecolors='black', linewidths=1.5, ax=ax_arr[i])
            elif data["type"] == "connection":
                nx.draw_networkx_nodes(G, pos, nodelist=[node], node_shape='d', node_color='lightpink', edgecolors='black', linewidths=1.5, ax=ax_arr[i])

        nx.draw_networkx_edges(G, pos, arrowstyle='-|>', arrowsize=10, edge_color='gray', ax=ax_arr[i])

        edge_labels = nx.get_edge_attributes(G, "action")
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8, label_pos=0.5, ax=ax_arr[i])


        nx.draw_networkx_labels(G, pos, labels={node: node for node, data in G.nodes(data=True)}, font_size=9, ax=ax_arr[i])

        ax_arr[i].set_axis_off()
        ax_arr[i].set_title(nombres[i])
    plt.show()

In [163]:
"""
G: Base Graph
G1: Eliminate node 0 and 2, connect everything to 1.
G2: Remove node 2, remove all connections except 1 from node 1 and set them to 0 and 8 (connection to 5 and 9). We add node 210 connected to 0
G3: We change node 2 (32) and create node 310. The connections from 0 to 3, 4 and 2 (changed, now it is 32) are maintained. We connect 0 to 9 (and 9 is disconnected from 32). Nodes 5 and 6 are eliminated.
G4: We eliminate node 2,3,5. Create node 410
G5: Subgraph of connections of 1 (minus node 0). Delete nodes 0,2,3,4.
G6: Subnetwork of connections of 0 and 2: Delete nodes 6,7,8,9.
G7: Same structure rename all nodes.
G8: Same structure change nodes 2,4,7,8
G9: Delete nodes 8 and 9
G10: We eliminate node 1 and all the actions are done by node 0 except 2 which is still connected to 4 and 5.
"""

def create_graphs():
  # Base Example --> Malicious attack

  G = nx.DiGraph()

  G.add_node(0, name="mw.exe", type="process")
  G.add_node(1, name="cmd.exe(1)", type="process")
  G.add_node(2, name="cmd.exe(2)", type="process")
  G.add_node(3, name="config.sys", type="file")
  G.add_node(4, name="IP1", type="connection")
  G.add_node(5, name="temp.tmp", type="file")
  G.add_node(6, name="user_data.db", type="file")
  G.add_node(7, name="browser_history.log", type="file")
  G.add_node(8, name="explorer.exe", type="process")
  G.add_node(9, name="IP2", type="connection")

  G.add_edges_from([
      (0, 1, {"action": "create"}),         # mw.exe creates the first cmd.exe process
      (0, 2, {"action": "create"}),        # mw.exe creates the second cmd.exe process
      (0, 3, {"action": "read"}),           # mw.exe reads the config.sys file
      (0, 4, {"action": "connect"}),          # mw.exe connects to IP1
      (1, 5, {"action": "create"}),          # cmd.exe(1) creates temp.tmp
      (1, 6, {"action": "read"}),           # cmd.exe(1) reads the user_data.db
      (1, 7, {"action": "read"}),           # cmd.exe(1) reads the browser_history.log
      (1, 8, {"action": "create"}),          # cmd.exe(1) creates the explorer.exe
      (1, 9, {"action": "connect"}),      # cmd.exe(1) connects to IP2
      (2, 5, {"action": "read"}),         # cmd.exe (2) reads temp.tmp
      (2, 4, {"action": "send"}),        # cmd.exe (2) sends data (temp.tmp) IP1
  ])


  G1 = nx.DiGraph()

  G1.add_node(1, name="cmd.exe(1)", type="process")
  G1.add_node(3, name="config.sys", type="file")
  G1.add_node(4, name="IP1", type="connection")
  G1.add_node(5, name="temp.tmp", type="file")
  G1.add_node(6, name="user_data.db", type="file")
  G1.add_node(7, name="browser_history.log", type="file")
  G1.add_node(8, name="explorer.exe", type="process")
  G1.add_node(9, name="IP2", type="connection")

  G1.add_edges_from([
      (1, 3, {"action": "read"}),      # cmd.exe reads config.sys
      (1, 4, {"action": "connect"}),   # cmd.exe connects to IP1
      (1, 5, {"action": "create"}),    # cmd.exe creates temp.tmp
      (1, 6, {"action": "read"}),      # cmd.exe reads user_data.db
      (1, 7, {"action": "read"}),      # cmd.exe reads browser_history.log
      (1, 8, {"action": "create"}),    # cmd.exe creates explorer.exe
      (1, 9, {"action": "connect"}),   # cmd.exe connects to IP2
  ])


  G2 = nx.DiGraph()

  G2.add_node(0, name="cmd.exe(new)", type="process") # Watch out! new cmd process!
  G2.add_node(1, name="cmd.exe(1)", type="process")
  G2.add_node(3, name="config.sys", type="file")
  G2.add_node(4, name="IP1", type="connection")
  G2.add_node(5, name="temp.tmp", type="file")
  G2.add_node(6, name="user_data.db", type="file")
  G2.add_node(7, name="browser_history.log", type="file")
  G2.add_node(8, name="explorer.exe", type="process")
  G2.add_node(9, name="IP2", type="connection")
  G2.add_node(210, name="data.enc", type="file")

  G2.add_edges_from([
      (0, 1, {"action": "create"}),
      (1, 7, {"action": "open"}),        
      (0, 210, {"action": "create"}),      
      (0, 4, {"action": "send"}),           
      (0, 6, {"action": "read"}),         
      (0, 3, {"action": "read"}),         
      (0, 8, {"action": "create"}),           
      (8, 9, {"action": "connect"}),           
      (8, 5, {"action": "create"}),      
  ])

  G3 = nx.DiGraph()

  G3.add_node(0, name="mw.exe", type="process")
  G3.add_node(1, name="cmd.exe(1)", type="process")
  G3.add_node(32, name="cmd.exe(2)(new)", type="process")
  G3.add_node(3, name="config.sys", type="file")
  G3.add_node(6, name="user_data.db", type="file")
  G3.add_node(4, name="IP1", type="connection")
  G3.add_node(7, name="browser_history.log", type="file")
  G3.add_node(9, name="IP2", type="connection")
  G3.add_node(310, name="doc.docx(new)", type="file")

  G3.add_edges_from([
      (0, 1, {"action": "create"}),         # mw.exe creates the first cmd.exe process
      (0, 3, {"action": "read"}),           # mw.exe reads the config.sys file
      (0, 4, {"action": "connect"}),          # mw.exe connects to IP1
      (0, 9, {"action": "connect"}),          # mw.exe connects to IP2
      (0, 32, {"action": "create"}),      #
      (32, 310, {"action": "create"}),      #
      (32, 7, {"action": "read"}),
      (1, 7, {"action": "read"}),
      (1, 6, {"action": "read"}),
      (9, 0, {"action": "receive"}),
  ])

  G4 = nx.DiGraph()

  G4.add_node(0, name="mw.exe", type="process")
  G4.add_node(1, name="cmd.exe(1)", type="process")
  G4.add_node(4, name="IP1", type="connection")
  G4.add_node(6, name="user_data.db", type="file")
  G4.add_node(7, name="browser_history.log", type="file")
  G4.add_node(8, name="explorer.exe", type="process")
  G4.add_node(9, name="IP2", type="connection")
  G4.add_node(410, name="encrypted_file.enc", type="file")

  G4.add_edges_from([
      (0, 1, {"action": "create"}),         # mw.exe creates the first cmd.exe process
      (0, 4, {"action": "connect"}),          # mw.exe connects to IP1
      (4, 0, {"action": "receive"}),
      (0, 410, {"action": "create"}),
      (1, 6, {"action": "read"}),           # cmd.exe(1) reads the user_data.db
      (1, 7, {"action": "read"}),           # cmd.exe(1) reads the browser_history.log
      (1, 8, {"action": "create"}),          # cmd.exe(1) creates the explorer.exe
      (1, 9, {"action": "connect"}),
  ])


  G5 = nx.DiGraph()

  G5.add_node(1, name="cmd.exe(1)", type="process")
  G5.add_node(5, name="temp.tmp", type="file")
  G5.add_node(6, name="user_data.db", type="file")
  G5.add_node(7, name="browser_history.log", type="file")
  G5.add_node(8, name="explorer.exe", type="process")
  G5.add_node(9, name="IP2", type="connection")

  G5.add_edges_from([
      (1, 5, {"action": "create"}),          # cmd.exe(1) creates temp.tmp
      (1, 6, {"action": "read"}),           # cmd.exe(1) reads the user_data.db
      (1, 7, {"action": "read"}),           # cmd.exe(1) reads the browser_history.log
      (1, 8, {"action": "create"}),          # cmd.exe(1) creates the explorer.exe
      (1, 9, {"action": "connect"}),      # cmd.exe(1) connects to IP2
  ])



  G6 = nx.DiGraph()

  G6.add_node(0, name="mw.exe", type="process")
  G6.add_node(1, name="cmd.exe(1)", type="process")
  G6.add_node(2, name="cmd.exe(2)", type="process")
  G6.add_node(3, name="config.sys", type="file")
  G6.add_node(4, name="IP1", type="connection")
  G6.add_node(5, name="temp.tmp", type="file")

  G6.add_edges_from([
      (0, 1, {"action": "create"}),         # mw.exe creates the first cmd.exe process
      (0, 2, {"action": "create"}),        # mw.exe creates the second cmd.exe process
      (0, 3, {"action": "read"}),           # mw.exe reads the config.sys file
      (0, 4, {"action": "connect"}),          # mw.exe connects to IP1
      (1, 5, {"action": "create"}),          # cmd.exe(1) creates temp.tmp
      (2, 5, {"action": "read"}),         # cmd.exe (2) reads temp.tmp
      (2, 4, {"action": "send"}),        # cmd.exe (2) sends data (temp.tmp) IP1
  ])

  G7 = nx.DiGraph()

  G7.add_node(70, name="mw.exe(new)", type="process")
  G7.add_node(71, name="cmd.exe(1)(new)", type="process")
  G7.add_node(72, name="cmd.exe(2)(new)", type="process")
  G7.add_node(73, name="config.sys(new)", type="file")
  G7.add_node(74, name="IP1(new)", type="connection")
  G7.add_node(75, name="temp.tmp(new)", type="file")
  #G7.add_node(75, name="temp.tmp(new)", type="connection")
  G7.add_node(76, name="user_data.db(new)", type="file")
  G7.add_node(77, name="browser_history.log(new)", type="file")
  G7.add_node(78, name="explorer.exe(new)", type="process")
  G7.add_node(79, name="IP2(new)", type="connection")

  G7.add_edges_from([
      (70, 71, {"action": "create"}),         # mw.exe creates the first cmd.exe process
      (70, 72, {"action": "create"}),        # mw.exe creates the second cmd.exe process
      (70, 73, {"action": "read"}),           # mw.exe reads the config.sys file
      (70, 74, {"action": "connect"}),          # mw.exe connects to IP1
      (71, 75, {"action": "create"}),          # cmd.exe(1) creates temp.tmp
      (71, 76, {"action": "read"}),           # cmd.exe(1) reads the user_data.db
      (71, 77, {"action": "read"}),           # cmd.exe(1) reads the browser_history.log
      (71, 78, {"action": "create"}),          # cmd.exe(1) creates the explorer.exe
      (71, 79, {"action": "connect"}),      # cmd.exe(1) connects to IP2
      (72, 75, {"action": "read"}),         # cmd.exe (2) reads temp.tmp
      (72, 74, {"action": "send"}),        # cmd.exe (2) sends data (temp.tmp) IP1
  ])


  G8 = nx.DiGraph()

  G8.add_node(0, name="mw.exe", type="process")
  G8.add_node(1, name="cmd.exe(1)", type="process")
  G8.add_node(82, name="cmd.exe(2)(new)", type="process")
  G8.add_node(3, name="config.sys", type="file")
  G8.add_node(84, name="IP1(new)", type="connection")
  G8.add_node(5, name="temp.tmp", type="file")
  G8.add_node(6, name="user_data.db", type="file")
  G8.add_node(87, name="browser_history.log(new)", type="file")
  G8.add_node(88, name="explorer.exe(new)", type="process")
  G8.add_node(9, name="IP2", type="connection")

  G8.add_edges_from([
      (0, 1, {"action": "create"}),         # mw.exe creates the first cmd.exe process
      (0, 82, {"action": "create"}),        # mw.exe creates the second cmd.exe process
      (0, 3, {"action": "read"}),           # mw.exe reads the config.sys file
      (0, 84, {"action": "connect"}),          # mw.exe connects to IP1
      (1, 5, {"action": "create"}),          # cmd.exe(1) creates temp.tmp
      (1, 6, {"action": "read"}),           # cmd.exe(1) reads the user_data.db
      (1, 87, {"action": "read"}),           # cmd.exe(1) reads the browser_history.log
      (1, 88, {"action": "create"}),          # cmd.exe(1) creates the explorer.exe
      (1, 9, {"action": "connect"}),      # cmd.exe(1) connects to IP2
      (82, 5, {"action": "read"}),         # cmd.exe (2) reads temp.tmp
      (82, 84, {"action": "send"}),        # cmd.exe (2) sends data (temp.tmp) IP1
  ])


  G9 = nx.DiGraph()

  G9.add_node(0, name="mw.exe", type="process")
  G9.add_node(1, name="cmd.exe(1)", type="process")
  G9.add_node(2, name="cmd.exe(2)", type="process")
  G9.add_node(3, name="config.sys", type="file")
  G9.add_node(4, name="IP1", type="connection")
  G9.add_node(5, name="temp.tmp", type="file")
  G9.add_node(6, name="user_data.db", type="file")
  G9.add_node(7, name="browser_history.log", type="file")

  G9.add_edges_from([
      (0, 1, {"action": "create"}),         # mw.exe creates the first cmd.exe process
      (0, 2, {"action": "create"}),        # mw.exe creates the second cmd.exe process
      (0, 3, {"action": "read"}),           # mw.exe reads the config.sys file
      (0, 4, {"action": "connect"}),          # mw.exe connects to IP1
      (1, 5, {"action": "create"}),          # cmd.exe(1) creates temp.tmp
      (1, 6, {"action": "read"}),           # cmd.exe(1) reads the user_data.db
      (1, 7, {"action": "read"}),           # cmd.exe(1) reads the browser_history.log
      (2, 5, {"action": "read"}),         # cmd.exe (2) reads temp.tmp
      (2, 4, {"action": "send"}),        # cmd.exe (2) sends data (temp.tmp) IP1
  ])

  G10 = nx.DiGraph()

  G10.add_node(0, name="mw.exe", type="process")
  G10.add_node(2, name="cmd.exe(2)", type="process")
  G10.add_node(3, name="config.sys", type="file")
  G10.add_node(4, name="IP1", type="connection")
  G10.add_node(5, name="temp.tmp", type="file")
  G10.add_node(6, name="user_data.db", type="file")
  G10.add_node(7, name="browser_history.log", type="file")
  G10.add_node(8, name="explorer.exe", type="process")
  G10.add_node(9, name="IP2", type="connection")

  G10.add_edges_from([
      (0, 2, {"action": "create"}),        # mw.exe creates the second cmd.exe process
      (0, 3, {"action": "read"}),           # mw.exe reads the config.sys file
      (0, 4, {"action": "connect"}),          # mw.exe connects to IP1
      (0, 5, {"action": "create"}),          # cmd.exe(1) creates temp.tmp
      (0, 6, {"action": "read"}),           # cmd.exe(1) reads the user_data.db
      (0, 7, {"action": "read"}),           # cmd.exe(1) reads the browser_history.log
      (0, 8, {"action": "create"}),          # cmd.exe(1) creates the explorer.exe
      (0, 9, {"action": "connect"}),      # cmd.exe(1) connects to IP2
      (2, 5, {"action": "read"}),         # cmd.exe (2) reads temp.tmp
      (2, 4, {"action": "send"}),        # cmd.exe (2) sends data (temp.tmp) IP1
  ])


  G11 = nx.DiGraph()

  G11.add_node(0, name="1", type="process")
  G11.add_node(0, name="2", type="process")
  G11.add_node(0, name="3", type="process")
  G11.add_node(0, name="4", type="process")
  G11.add_node(0, name="5", type="process")
  G11.add_node(0, name="6", type="process")

  return G, G1, G2, G3, G4, G5, G6, G7, G8, G9, G10,
  return G, G5, G6, G7, G8, G9

In [164]:
# Assigns a different type to each id
ids_as_types = True

In [165]:
path_graphs_gml = "SDTED/Data/CUSTOM_gml/"
path_graphs = "SDTED/Data/CUSTOM/"

arr_graphs = list(create_graphs())
"""
type_dict = {}
for g in arr_graphs:
  d = nx.get_node_attributes(g, "type")
  for k in d.keys():
    type_dict[k] = d[k]"""
for index, graph in enumerate(arr_graphs):
  nx.write_gml(graph, path_graphs_gml + "graph_" + str(index) + ".gml" )
# The id of each graph corresponds to its index in the array arr_graph

In [167]:
# First of all, I have to perform a re-assignment of node ids, because with this methodology each node id is unique, even if they are from different graphs
arr_nodes = []
arr_graphs_aux = []
# 1 because it has to start with 1 to correspond to the 1st line
i = 1
mapping = {}
type_dict = {}

# We use it to store the history of all ids correspondences
mapping_inverse = {}
for index, graph in enumerate(arr_graphs):

    for node in graph.nodes():
        mapping[node] = i
        mapping_inverse[i] = node
        d = nx.get_node_attributes(graph, "type")
        type_dict[i] = d[node]
        i = i + 1
    H = nx.relabel_nodes(graph, mapping)
    if ids_as_types:
        dict_graph = {}
        for node in H.nodes():
            dict_graph[node] = node
        nx.set_node_attributes(H, dict_graph, name="type")
    arr_graphs_aux.append(H)

arr_graphs = arr_graphs_aux   

In [168]:
#graph_indicator.txt
# column vector of graph identifiers for all nodes of all graphs,
# the value in the i-th line is the graph_id of the node with node_id i

open(path_graphs + "CUSTOM_graph_indicator.txt", "w")
for index, graph in enumerate(arr_graphs):
    with open(path_graphs + "CUSTOM_graph_indicator.txt", "a") as f:
        for node in graph.nodes():
            # Index must start with 1
            f.write(str(index + 1) + '\n')

In [169]:
#A.txt
#sparse (block diagonal) adjacency matrix for all graphs,
#each line corresponds to (row, col) resp. (node_id, node_id)
open(path_graphs + "CUSTOM_A.txt", "w")
for index, graph in enumerate(arr_graphs):
    with open(path_graphs + "CUSTOM_A.txt", "a") as f:
        for edge in graph.edges():
            # The i-th node corresponds to the i-th row (starting with 1).
            f.write(str(edge[0]) + ', ' + str(edge[1] ) + '\n')

In [170]:
#edge_labels.txt*
#(m lines; same size as DS_A_sparse.txt)
#labels for the edges in DD_A_sparse.txt 
DICT_EDGE_TYPE = {
    "create": 0,
    "read": 1,
    "open": 2,
    "connect": 3,
    "send": 4, 
    "receive": 5 
}
open(path_graphs + "CUSTOM_edge_labels.txt", "w")
for index, graph in enumerate(arr_graphs):
    with open(path_graphs + "CUSTOM_edge_labels.txt", "a") as f:
        actions = nx.get_edge_attributes(graph, 'action')
        for edge in graph.edges():
            f.write(str(DICT_EDGE_TYPE[actions[(edge[0], edge[1])]]) + '\n')


In [171]:
#node_labels.txt
#column vector of node labels,
#the value in the i-th line corresponds to the node with node_id i

DICT_NODE_TYPE = {
    "file": 0,
    "process": 1,
    "connection": 2
}
open(path_graphs + "CUSTOM_node_labels.txt", "w")
for index, graph in enumerate(arr_graphs):
    with open(path_graphs + "CUSTOM_node_labels.txt", "a") as f:
        types = nx.get_node_attributes(graph, "type")
        for node in graph.nodes():
            if ids_as_types:
                f.write(str(types[node]) + '\n')
            else:
                f.write(str(DICT_NODE_TYPE[types[node]]) + '\n')


# New file type that stores the types of each id (to calculate the initial cost matrix)
# The file consists of a dictionary separated with commas.  First col = id, second col = type
if ids_as_types:
    path_file = path_graphs + "CUSTOM_node_types.txt"
    with open(path_file, "w") as f:
        for key_dict in type_dict.keys():
            f.write(str(key_dict) + ',' + str(DICT_NODE_TYPE[type_dict[key_dict]]) + '\n')
    
    path_file2 = path_graphs + "CUSTOM_node_ids.txt"
    with open(path_file2, "w") as f:
        for key_dict in mapping_inverse.keys():
            f.write(str(key_dict) + ',' + str(mapping_inverse[key_dict]) + '\n')


In [172]:
#graph_labels.txt
#class labels for all graphs in the dataset,
#the value in the i-th line is the class label of the graph with graph_id i

with open(path_graphs + "CUSTOM_graph_labels.txt", "w") as f:
    for index, graph in enumerate(arr_graphs):
        f.write("1\n")